In [42]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["GROQ_API_KEY"]

In [43]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(model="qwen/qwen3.8-27b")

## RunnablePassthrough
* It does not do anything to the input data.
* Let's see it in a very simple example: a chain with just RunnablePassthrough() will output the original input without any modification.

In [44]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

In [45]:
chain.invoke("badrawy")

'badrawy'

## RunnableLambda
* To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.
* Let's define a very simple function to create Russian lastnames:

In [46]:
def russian_lastname(name: str) -> str:
    return f"{name}ovich"

In [47]:
from langchain_core.runnables import RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_lastname)

In [48]:
chain.invoke("badrawy")

'badrawyovich'

## RunnableParallel
* We will use RunnableParallel() for running tasks in parallel.
* This is probably the most important and most useful Runnable from LangChain.
* In the following chain, RunnableParallel is going to run these two tasks in parallel:
    * operation_a will use RunnablePassthrough.
    * operation_b will use RunnableLambda with the russian_lastname function.

In [49]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": RunnableLambda(russian_lastname)
    }
)

In [50]:
chain.invoke("Bappy")

{'operation_a': 'Bappy', 'operation_b': 'Bappyovich'}

In [51]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [52]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [53]:
def russian_lastname_from_dictionary(person):
    return person["name"] + "ovich"

In [54]:
chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "soccer_player": RunnableLambda(russian_lastname_from_dictionary),
        "operation_c": RunnablePassthrough(),
    }
) | prompt | chatModel | output_parser

In [55]:
chain.invoke({
    "name1": "Jordam",
    "name": "Abram"
})

'A particularly curious fact about Roman Abramovich is that **he did not inherit his wealth from a state-owned corporation, but rather built his empire by buying and reselling the rights to privatize Soviet-era state assets.**\n\nHere’s the breakdown of why this is so unusual:\n\n1. **The "Shadow" of the State:** In the 1990s, as the Russian economy was being privatized, Abramovich, then a young businessman in Magadan, formed a partnership with the local governor, Nikolay Khodorkovsky (who later became the head of Yukos). They created a company called **Syndicate** to manage the privatization of state-owned enterprises in the Far East.\n2. **Buying the "Rights" to Buy:** Instead of just buying companies, Abramovich and his partners bought the *rights* to privatize state assets. This meant they could choose which companies to acquire, often at steep discounts, and then resell those companies or their assets at much higher prices.\n3. **The Siberian Aluminum Deal:** One of his first majo

## Let's see a more advanced use of RunnableParallel

In [56]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_huggingface import HuggingFaceEmbeddings

# Standard, fast, and lightweight embedding chatModel
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=embeddings_model
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)


retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | chatModel
    | StrOutputParser()
)

retrieval_chain.invoke("What is dswithbappy?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4649.30it/s]


'Based on the provided context, **dswithbappy** is a source that focuses on providing content in English related to Data Science, AI, ML, DL, CV, NLP, and Python programming.'

In [57]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=embeddings_model
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | chatModel
    | StrOutputParser()
)

chain.invoke({"question": "What is dswithbappy?", "language": "Pirate English"})

"Arr, dswithbappy be a vessel focused on providin' the booty of Data Science, AI, ML, DL, CV, NLP, and Python programming, all spoken in the tongue of England, matey."